In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error

from xgboost import XGBRegressor

In [2]:
DATA_PATH = Path(
    "../data/processed/hdb_resale_clean.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["month"]
)

print(
    "Rows:",
    len(df)
)

print(
    "Date range:",
    df["month"].min(),
    "to",
    df["month"].max()
)

Rows: 244230
Date range: 2017-01-01 00:00:00 to 2026-11-01 00:00:00


In [3]:
development_df = df[
    df["month"]
    <= "2026-08-01"
].copy()

calibration_df = df[
    df["month"]
    == "2026-09-01"
].copy()

print(
    "Development rows:",
    len(development_df)
)

print(
    "Development period:",
    development_df["month"].min(),
    "to",
    development_df["month"].max()
)

print()

print(
    "Calibration rows:",
    len(calibration_df)
)

print(
    "Calibration period:",
    calibration_df["month"].min(),
    "to",
    calibration_df["month"].max()
)

Development rows: 239458
Development period: 2017-01-01 00:00:00 to 2026-08-01 00:00:00

Calibration rows: 2239
Calibration period: 2026-09-01 00:00:00 to 2026-09-01 00:00:00


In [4]:
FEATURES = [
    "town",
    "flat_type",
    "storey_mid",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease_months",
    "month_index"
]

TARGET = "resale_price"

X_development = (
    development_df[
        FEATURES
    ].copy()
)

y_development = (
    development_df[
        TARGET
    ].copy()
)

X_calibration = (
    calibration_df[
        FEATURES
    ].copy()
)

y_calibration = (
    calibration_df[
        TARGET
    ].copy()
)

print(
    "Development:",
    X_development.shape
)

print(
    "Calibration:",
    X_calibration.shape
)

Development: (239458, 8)
Calibration: (2239, 8)


In [5]:
categorical_features = [
    "town",
    "flat_type",
    "flat_model"
]

numeric_features = [
    "storey_mid",
    "floor_area_sqm",
    "lease_commence_date",
    "remaining_lease_months",
    "month_index"
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)


calibration_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=8,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

calibration_model.fit(
    X_development,
    y_development
)

print(
    "XGBoost refitted through:",
    development_df["month"].max()
)

XGBoost refitted through: 2026-08-01 00:00:00


In [6]:
categorical_features = [
    "town",
    "flat_type",
    "flat_model"
]

numeric_features = [
    "storey_mid",
    "floor_area_sqm",
    "lease_commence_date",
    "remaining_lease_months",
    "month_index"
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)


calibration_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=8,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                n_jobs=-1,
                random_state=42
            )
        )
    ]
)

calibration_model.fit(
    X_development,
    y_development
)

print(
    "XGBoost refitted through:",
    development_df["month"].max()
)

XGBoost refitted through: 2026-08-01 00:00:00


In [7]:
calibration_predictions = (
    calibration_model.predict(
        X_calibration
    )
)

calibration_results = (
    calibration_df.copy()
)

calibration_results[
    "predicted_price"
] = calibration_predictions

calibration_results[
    "absolute_error"
] = np.abs(
    calibration_results[
        "resale_price"
    ]
    - calibration_results[
        "predicted_price"
    ]
)

calibration_results[
    "ape"
] = (
    calibration_results[
        "absolute_error"
    ]
    / calibration_results[
        "resale_price"
    ]
    * 100
)

print(
    "Calibration MAE:",
    round(
        calibration_results[
            "absolute_error"
        ].mean(),
        2
    )
)

print(
    "Calibration Median APE:",
    round(
        calibration_results[
            "ape"
        ].median(),
        4
    ),
    "%"
)

Calibration MAE: 33202.51
Calibration Median APE: 3.6994 %


In [8]:
def score_comparables(
    target_row,
    candidates
):
    """
    Score historical transactions according to
    similarity with the target flat.

    Higher score = more similar.
    """

    scored = candidates.copy()

    # Same block
    scored["same_block"] = (
        (scored["block"] == target_row["block"])
        & (scored["street_name"] == target_row["street_name"])
    ).astype(int)

    # Same street
    scored["same_street"] = (
        scored["street_name"]
        == target_row["street_name"]
    ).astype(int)

    # Same flat model
    scored["same_model"] = (
        scored["flat_model"]
        == target_row["flat_model"]
    ).astype(int)

    # Floor-area difference
    scored["area_diff"] = np.abs(
        scored["floor_area_sqm"]
        - target_row["floor_area_sqm"]
    )

    # Remaining-lease difference
    scored["lease_diff"] = np.abs(
        scored["remaining_lease_months"]
        - target_row["remaining_lease_months"]
    )

    # Storey difference
    scored["storey_diff"] = np.abs(
        scored["storey_mid"]
        - target_row["storey_mid"]
    )

    # Months between target and transaction
    scored["months_ago"] = (
        (
            target_row["month"].year
            - scored["month"].dt.year
        ) * 12
        +
        (
            target_row["month"].month
            - scored["month"].dt.month
        )
    )

    # ------------------------------------------
    # Similarity components, each scaled 0–1
    # ------------------------------------------

    scored["area_score"] = (
        1
        - scored["area_diff"] / 20
    ).clip(
        lower=0,
        upper=1
    )

    scored["lease_score"] = (
        1
        - scored["lease_diff"] / 120
    ).clip(
        lower=0,
        upper=1
    )

    scored["storey_score"] = (
        1
        - scored["storey_diff"] / 15
    ).clip(
        lower=0,
        upper=1
    )

    scored["recency_score"] = (
        1
        - scored["months_ago"] / 12
    ).clip(
        lower=0,
        upper=1
    )

    # ------------------------------------------
    # Overall score, scaled to 0–100
    # ------------------------------------------

    scored["comparable_score"] = (
        scored["same_block"] * 20
        + scored["same_street"] * 15
        + scored["same_model"] * 10
        + scored["area_score"] * 20
        + scored["lease_score"] * 15
        + scored["storey_score"] * 10
        + scored["recency_score"] * 10
    )

    return (
        scored
        .sort_values(
            "comparable_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

In [9]:
def get_comparable_candidates(
    target_row,
    history_df
):
    """
    Return same-town, same-flat-type transactions
    from the 12 months before the target month.
    """

    target_month = target_row["month"]

    start_month = (
        target_month
        - pd.DateOffset(months=12)
    )

    candidates = history_df[
        (
            history_df["town"]
            == target_row["town"]
        )
        &
        (
            history_df["flat_type"]
            == target_row["flat_type"]
        )
        &
        (
            history_df["month"]
            < target_month
        )
        &
        (
            history_df["month"]
            >= start_month
        )
    ].copy()

    return candidates

In [10]:
target = calibration_df.iloc[0]

candidates = get_comparable_candidates(
    target,
    development_df
)

scored_candidates = score_comparables(
    target,
    candidates
)

print(
    "Candidates:",
    len(scored_candidates)
)

scored_candidates[
    [
        "month",
        "town",
        "flat_type",
        "block",
        "street_name",
        "floor_area_sqm",
        "storey_range",
        "flat_model",
        "resale_price",
        "comparable_score"
    ]
].head(10)

Candidates: 18


,month,town,flat_type,block,street_name,floor_area_sqm,storey_range,flat_model,resale_price,comparable_score
0,2026-07-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44.0,10 TO 12,Improved,296000.0,91.958333
1,2026-07-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,44.0,04 TO 06,Improved,285000.0,59.708333
2,2026-08-01,ANG MO KIO,2 ROOM,323,ANG MO KIO AVE 3,44.0,04 TO 06,Improved,310000.0,59.416667
3,2026-04-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,44.0,04 TO 06,Improved,290000.0,57.583333
4,2026-08-01,ANG MO KIO,2 ROOM,323,ANG MO KIO AVE 3,48.0,01 TO 03,Improved,315000.0,57.541667
5,2026-03-01,ANG MO KIO,2 ROOM,508,ANG MO KIO AVE 8,44.0,04 TO 06,Improved,330000.0,55.500000
6,2026-04-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,44.0,07 TO 09,Improved,303000.0,55.458333
7,2025-12-01,ANG MO KIO,2 ROOM,323,ANG MO KIO AVE 3,44.0,04 TO 06,Improved,300000.0,53.875000
8,2025-11-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,44.0,01 TO 03,Improved,335000.0,53.666667
9,2026-02-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,44.0,10 TO 12,Improved,325000.0,52.166667


In [11]:
def calculate_evidence_summary(
    target_row,
    history_df
):
    """
    Calculate comparable evidence statistics
    for one target transaction.
    """

    candidates = get_comparable_candidates(
        target_row,
        history_df
    )

    if candidates.empty:
        return {
            "candidate_count": 0,
            "top5_avg_score": np.nan,
            "top5_strong_count": 0,
            "top1_score": np.nan
        }

    scored = score_comparables(
        target_row,
        candidates
    )

    top5 = scored.head(5)

    return {
        "candidate_count": len(scored),
        "top5_avg_score": top5[
            "comparable_score"
        ].mean(),
        "top5_strong_count": (
            top5["comparable_score"]
            >= 75
        ).sum(),
        "top1_score": scored[
            "comparable_score"
        ].iloc[0]
    }

In [12]:
test_summary = calculate_evidence_summary(
    target,
    development_df
)

test_summary

{'candidate_count': 18,
 'top5_avg_score': np.float64(65.24166666666666),
 'top5_strong_count': np.int64(1),
 'top1_score': np.float64(91.95833333333333)}

In [13]:
evidence_summaries = []

for index, target_row in calibration_df.iterrows():

    summary = calculate_evidence_summary(
        target_row,
        development_df
    )

    summary["index"] = index

    evidence_summaries.append(
        summary
    )

evidence_df = pd.DataFrame(
    evidence_summaries
).set_index(
    "index"
)

print(
    "Evidence rows:",
    len(evidence_df)
)

evidence_df.head()

Evidence rows: 2239


,candidate_count,top5_avg_score,top5_strong_count,top1_score
index,,,,
222067,18,65.241667,1,91.958333
222075,18,83.450000,3,95.166667
222082,18,62.366667,0,63.708333
222083,18,89.641667,5,96.083333
222087,18,26.500000,0,28.166667


In [14]:
calibration_results = (
    calibration_results
    .join(
        evidence_df
    )
)

calibration_results[
    [
        "resale_price",
        "predicted_price",
        "absolute_error",
        "candidate_count",
        "top5_avg_score",
        "top5_strong_count",
        "top1_score"
    ]
].head()

,resale_price,predicted_price,absolute_error,candidate_count,top5_avg_score,top5_strong_count,top1_score
222067,277000.0,301654.56250,24654.56250,18,65.241667,1,91.958333
222075,285000.0,315679.00000,30679.00000,18,83.450000,3,95.166667
222082,303000.0,309511.81250,6511.81250,18,62.366667,0,63.708333
222083,360000.0,340067.84375,19932.15625,18,89.641667,5,96.083333
222087,518000.0,618678.00000,100678.00000,18,26.500000,0,28.166667


In [15]:
calibration_results[
    [
        "candidate_count",
        "top5_avg_score",
        "top5_strong_count",
        "top1_score"
    ]
].describe()

,candidate_count,top5_avg_score,top5_strong_count,top1_score
count,2239.000000,2239.000000,2239.000000,2239.000000
mean,401.369808,78.751617,3.066548,88.371870
std,255.331362,8.999321,1.783787,9.457270
min,2.000000,26.500000,0.000000,28.166667
25%,191.000000,73.708333,2.000000,83.312500
50%,371.000000,78.950000,3.000000,91.500000
75%,579.000000,84.716667,5.000000,95.166667
max,895.000000,97.308333,5.000000,99.166667


In [16]:
print(
    "No candidates:",
    (
        calibration_results[
            "candidate_count"
        ]
        == 0
    ).sum()
)

print(
    "Fewer than 5 candidates:",
    (
        calibration_results[
            "candidate_count"
        ]
        < 5
    ).sum()
)

No candidates: 0
Fewer than 5 candidates: 2


In [17]:
threshold_results = []

for avg_threshold in [
    60,
    65,
    70,
    75
]:

    for strong_threshold in [
        1,
        2,
        3
    ]:

        accepted = (
            (
                calibration_results[
                    "candidate_count"
                ] >= 5
            )
            &
            (
                calibration_results[
                    "top5_avg_score"
                ] >= avg_threshold
            )
            &
            (
                calibration_results[
                    "top5_strong_count"
                ] >= strong_threshold
            )
        )

        accepted_df = (
            calibration_results[
                accepted
            ]
        )

        if len(accepted_df) == 0:
            continue

        threshold_results.append(
            {
                "avg_threshold":
                    avg_threshold,

                "strong_threshold":
                    strong_threshold,

                "accepted_count":
                    len(accepted_df),

                "accepted_pct":
                    len(accepted_df)
                    / len(
                        calibration_results
                    )
                    * 100,

                "mae":
                    accepted_df[
                        "absolute_error"
                    ].mean(),

                "median_ape_pct":
                    accepted_df[
                        "ape"
                    ].median()
            }
        )

threshold_results_df = (
    pd.DataFrame(
        threshold_results
    )
)

threshold_results_df

,avg_threshold,strong_threshold,accepted_count,accepted_pct,mae,median_ape_pct
0,60,1,2004,89.504243,31943.451363,3.672730
1,60,2,1683,75.167485,30942.517120,3.561461
2,60,3,1342,59.937472,30032.309776,3.499422
3,65,1,1968,87.896382,31689.017515,3.618216
4,65,2,1678,74.944172,30939.807695,3.559536
5,65,3,1342,59.937472,30032.309776,3.499422
6,70,1,1869,83.474766,31351.689088,3.610754
7,70,2,1655,73.916927,30876.975925,3.561461
8,70,3,1341,59.892809,30040.066625,3.501195
9,75,1,1557,69.539973,30012.367754,3.533316


In [18]:
threshold_results_df.sort_values(
    [
        "accepted_pct",
        "mae"
    ],
    ascending=[
        False,
        True
    ]
)

,avg_threshold,strong_threshold,accepted_count,accepted_pct,mae,median_ape_pct
0,60,1,2004,89.504243,31943.451363,3.672730
3,65,1,1968,87.896382,31689.017515,3.618216
6,70,1,1869,83.474766,31351.689088,3.610754
1,60,2,1683,75.167485,30942.517120,3.561461
4,65,2,1678,74.944172,30939.807695,3.559536
7,70,2,1655,73.916927,30876.975925,3.561461
9,75,1,1557,69.539973,30012.367754,3.533316
10,75,2,1511,67.485485,29899.153851,3.507437
2,60,3,1342,59.937472,30032.309776,3.499422
5,65,3,1342,59.937472,30032.309776,3.499422


In [19]:
threshold_results = []

for candidate_threshold in [
    5,
    10,
    15
]:

    for avg_threshold in [
        65,
        70,
        75
    ]:

        for strong_threshold in [
            1,
            2,
            3
        ]:

            accepted = (
                (
                    calibration_results[
                        "candidate_count"
                    ] >= candidate_threshold
                )
                &
                (
                    calibration_results[
                        "top5_avg_score"
                    ] >= avg_threshold
                )
                &
                (
                    calibration_results[
                        "top5_strong_count"
                    ] >= strong_threshold
                )
            )

            accepted_df = (
                calibration_results[
                    accepted
                ]
            )

            if len(accepted_df) == 0:
                continue

            threshold_results.append(
                {
                    "candidate_threshold":
                        candidate_threshold,

                    "avg_threshold":
                        avg_threshold,

                    "strong_threshold":
                        strong_threshold,

                    "accepted_count":
                        len(accepted_df),

                    "accepted_pct":
                        len(accepted_df)
                        / len(calibration_results)
                        * 100,

                    "mae":
                        accepted_df[
                            "absolute_error"
                        ].mean(),

                    "median_ape_pct":
                        accepted_df[
                            "ape"
                        ].median()
                }
            )


threshold_results_df = (
    pd.DataFrame(
        threshold_results
    )
)

threshold_results_df

,candidate_threshold,avg_threshold,strong_threshold,accepted_count,accepted_pct,mae,median_ape_pct
0,5,65,1,1968,87.896382,31689.017515,3.618216
1,5,65,2,1678,74.944172,30939.807695,3.559536
2,5,65,3,1342,59.937472,30032.309776,3.499422
3,5,70,1,1869,83.474766,31351.689088,3.610754
4,5,70,2,1655,73.916927,30876.975925,3.561461
5,5,70,3,1341,59.892809,30040.066625,3.501195
6,5,75,1,1557,69.539973,30012.367754,3.533316
7,5,75,2,1511,67.485485,29899.153851,3.507437
8,5,75,3,1320,58.954891,29615.919508,3.463806
9,10,65,1,1966,87.807057,31637.052883,3.613737


In [20]:
candidate_rules = (
    threshold_results_df[
        threshold_results_df[
            "accepted_pct"
        ] >= 50
    ]
    .sort_values(
        [
            "mae",
            "accepted_pct"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)

candidate_rules

,candidate_threshold,avg_threshold,strong_threshold,accepted_count,accepted_pct,mae,median_ape_pct
0,15,75,3,1312,58.597588,29568.915801,3.463806
1,5,75,3,1320,58.954891,29615.919508,3.463806
2,10,75,3,1320,58.954891,29615.919508,3.463806
3,15,75,2,1502,67.083519,29838.695281,3.504316
4,5,75,2,1511,67.485485,29899.153851,3.507437
5,10,75,2,1511,67.485485,29899.153851,3.507437
6,15,75,1,1548,69.138008,29954.363978,3.532614
7,15,65,3,1334,59.580170,29988.578336,3.499422
8,15,70,3,1333,59.535507,29996.348931,3.501195
9,5,75,1,1557,69.539973,30012.367754,3.533316


In [21]:
EVIDENCE_RULE = {
    "min_candidate_count": 10,
    "min_top5_avg_score": 75,
    "min_top5_strong_count": 2,
    "strong_score_threshold": 75
}

EVIDENCE_RULE

{'min_candidate_count': 10,
 'min_top5_avg_score': 75,
 'min_top5_strong_count': 2,
 'strong_score_threshold': 75}

In [22]:
calibration_results["evidence_sufficient"] = (
    (
        calibration_results["candidate_count"]
        >= EVIDENCE_RULE["min_candidate_count"]
    )
    &
    (
        calibration_results["top5_avg_score"]
        >= EVIDENCE_RULE["min_top5_avg_score"]
    )
    &
    (
        calibration_results["top5_strong_count"]
        >= EVIDENCE_RULE["min_top5_strong_count"]
    )
)

In [23]:
accepted_count = (
    calibration_results[
        "evidence_sufficient"
    ].sum()
)

abstained_count = (
    (~calibration_results[
        "evidence_sufficient"
    ]).sum()
)

accepted_pct = (
    accepted_count
    / len(calibration_results)
    * 100
)

print(
    "Accepted:",
    accepted_count
)

print(
    "Abstained:",
    abstained_count
)

print(
    "Accepted %:",
    round(
        accepted_pct,
        2
    )
)

Accepted: 1511
Abstained: 728
Accepted %: 67.49


In [24]:
accepted_calibration = (
    calibration_results[
        calibration_results[
            "evidence_sufficient"
        ]
    ].copy()
)

print(
    "Accepted MAE:",
    round(
        accepted_calibration[
            "absolute_error"
        ].mean(),
        2
    )
)

print(
    "Accepted Median APE:",
    round(
        accepted_calibration[
            "ape"
        ].median(),
        4
    ),
    "%"
)

Accepted MAE: 29899.15
Accepted Median APE: 3.5074 %


In [25]:
accepted_calibration[
    "absolute_residual"
] = np.abs(
    accepted_calibration["resale_price"]
    - accepted_calibration["predicted_price"]
)

accepted_calibration[
    "absolute_residual"
].describe()

count      1511.000000
mean      29899.153851
std       30075.411693
min           2.937500
25%        9256.671875
50%       21991.562500
75%       40032.312500
max      298153.250000
Name: absolute_residual, dtype: float64

In [26]:
ALPHA = 0.10

n_calibration = len(
    accepted_calibration
)

quantile_level = min(
    1.0,
    np.ceil(
        (n_calibration + 1)
        * (1 - ALPHA)
    )
    / n_calibration
)

interval_margin = np.quantile(
    accepted_calibration[
        "absolute_residual"
    ],
    quantile_level,
    method="higher"
)

print(
    "Calibration cases:",
    n_calibration
)

print(
    "Quantile level:",
    round(
        quantile_level,
        4
    )
)

print(
    "90% interval margin:",
    round(
        interval_margin,
        2
    )
)

Calibration cases: 1511
Quantile level: 0.9007
90% interval margin: 65339.25


In [27]:
accepted_calibration[
    "interval_lower"
] = (
    accepted_calibration[
        "predicted_price"
    ]
    - interval_margin
)

accepted_calibration[
    "interval_upper"
] = (
    accepted_calibration[
        "predicted_price"
    ]
    + interval_margin
)

accepted_calibration[
    "interval_covered"
] = (
    (
        accepted_calibration[
            "resale_price"
        ]
        >= accepted_calibration[
            "interval_lower"
        ]
    )
    &
    (
        accepted_calibration[
            "resale_price"
        ]
        <= accepted_calibration[
            "interval_upper"
        ]
    )
)

In [28]:
calibration_coverage = (
    accepted_calibration[
        "interval_covered"
    ].mean()
    * 100
)

interval_width = (
    interval_margin
    * 2
)

print(
    "Calibration coverage:",
    round(
        calibration_coverage,
        2
    ),
    "%"
)

print(
    "Interval margin:",
    round(
        interval_margin,
        2
    )
)

print(
    "Interval width:",
    round(
        interval_width,
        2
    )
)

Calibration coverage: 90.14 %
Interval margin: 65339.25
Interval width: 130678.5


In [29]:
CALIBRATION_DIR = Path(
    "../models/final"
)

CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

calibration_metadata = {
    "calibration_month": "2026-09",
    "evidence_rule": EVIDENCE_RULE,
    "interval": {
        "method": "split_conformal_absolute_residual",
        "target_coverage": 0.90,
        "alpha": ALPHA,
        "calibration_cases": int(
            n_calibration
        ),
        "interval_margin": float(
            interval_margin
        ),
        "interval_width": float(
            interval_width
        ),
        "observed_calibration_coverage_pct": float(
            calibration_coverage
        )
    }
}

CALIBRATION_METADATA_PATH = (
    CALIBRATION_DIR
    / "hdb_resale_calibration.json"
)

with open(
    CALIBRATION_METADATA_PATH,
    "w"
) as file:

    json.dump(
        calibration_metadata,
        file,
        indent=4
    )

print(
    "Saved calibration metadata to:",
    CALIBRATION_METADATA_PATH
)

Saved calibration metadata to: ..\models\final\hdb_resale_calibration.json


In [30]:
threshold_results_df.to_csv(
    CALIBRATION_DIR
    / "evidence_threshold_results.csv",
    index=False
)

In [31]:
FINAL_MODEL_DIR = Path(
    "../models/final"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CALIBRATED_MODEL_PATH = (
    FINAL_MODEL_DIR
    / "hdb_resale_xgboost_calibrated.joblib"
)

calibration_model.fit(
    X_development,
    y_development
)

joblib.dump(
    calibration_model,
    CALIBRATED_MODEL_PATH,
    compress=3
)

print(
    "Saved calibrated model to:",
    CALIBRATED_MODEL_PATH
)

Saved calibrated model to: ..\models\final\hdb_resale_xgboost_calibrated.joblib
